# Notebook 5: Transfer Learning with a Pretrained ResNet

**Goal.** Classify 102 species of flowers with only **10 training images per class**, by reusing a network pretrained on ImageNet. Compare three strategies: training from scratch, feature extraction, and fine-tuning.

**What you will learn**

1. Why features learned on one dataset transfer to another, and which layers transfer best.
2. The ResNet architecture and residual connections, briefly.
3. Feature extraction (frozen backbone + new head) vs fine-tuning (update everything).
4. Practical details: ImageNet preprocessing, replacing the head, freezing parameters, BatchNorm in frozen networks, discriminative learning rates.
5. How to inspect the learned embedding space with t-SNE.
6. Saving a model and running inference on a single new image.

> **Runtime:** GPU strongly recommended. The dataset (~330 MB) downloads automatically from the Oxford VGG website.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt
import numpy as np
import time, copy

torch.manual_seed(0); np.random.seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## 1. Why transfer learning works

Training a deep CNN from scratch needs a lot of labelled data: ImageNet has 1.28 million images. Most real projects have hundreds or a few thousand. With 1,020 images and 11M parameters, a network trained from scratch will mostly memorise.

The key observation (Yosinski et al., 2014, "How transferable are features in deep neural networks?"; Razavian et al., 2014, "CNN Features off-the-shelf") is that CNN features are organised from **general** to **specific**:

* **Early layers** learn edges, colour blobs, textures (Notebook 4). These are useful for almost any natural-image task.
* **Middle layers** learn parts and motifs: still fairly generic.
* **Late layers** become specific to the original task's classes.

So a network trained on a large, diverse dataset is a powerful **feature extractor**. We keep its body (the **backbone**), throw away its ImageNet-specific classifier (the **head**), and train a new head for our classes.

Formally, the pretrained backbone gives a map $\phi: \mathbb{R}^{3\times224\times224}\to\mathbb{R}^{512}$. If $\phi$ already separates our classes well, even a **linear** classifier $g(z) = Wz + b$ on top of $\phi$ will do well, and a linear classifier on 512 features with 1,020 examples is a very manageable learning problem.

### Two strategies

| | Feature extraction | Fine-tuning |
|---|---|---|
| What is trained | only the new head | head + (some or all of) backbone |
| Trainable params | ~52k | ~11M |
| Data needed | very little | more |
| Risk | features might not fit the new domain | overfitting; destroying pretrained features with a large LR |
| Speed | fast | slower |

**Rule of thumb:** small dataset similar to ImageNet → feature extraction (or gentle fine-tuning). Larger dataset or very different domain (medical, satellite, microscopy) → fine-tune more layers.

## 2. A quick look at ResNet

Deep plain networks are hard to optimise: beyond ~20 layers, *training* error gets worse as depth increases. ResNet (He et al., 2016) fixes this with **residual connections**. Each block learns a residual $\mathcal{F}$ and adds its input back:

$$
y = \text{ReLU}\big(\mathcal{F}(x) + x\big), \qquad \mathcal{F}(x) = \text{BN}(\text{Conv}(\text{ReLU}(\text{BN}(\text{Conv}(x))))).
$$

If the optimal block is close to the identity, the network only needs to push $\mathcal{F}$ toward zero, which is easy. The identity path also gives gradients a direct route back to early layers.

ResNet-18 = a $7\times7$ stride-2 stem + max pool, four stages of 2 residual blocks each (64, 128, 256, 512 channels), global average pooling and a final `fc: Linear(512, 1000)`. That final `fc` is the only part we must replace.

In [ ]:
weights = models.ResNet18_Weights.IMAGENET1K_V1
backbone_demo = models.resnet18(weights=weights)
print(backbone_demo.fc)
for name, child in backbone_demo.named_children():
    n = sum(p.numel() for p in child.parameters())
    print(f"{name:<8} {n:>10,} params")

## 3. Data: Oxford Flowers-102

102 flower categories common in the UK. The official split is **10 images per class** for training (1,020), 10 per class for validation (1,020), and the remaining 6,149 for testing. This tiny training set is precisely the regime where transfer learning shines.

### Preprocessing for a pretrained model

The pretrained weights expect the **same input statistics** as during ImageNet training:

* RGB, scaled to $[0,1]$, then normalised with ImageNet mean $(0.485, 0.456, 0.406)$ and std $(0.229, 0.224, 0.225)$.
* Around $224\times224$ resolution.

Using different normalisation is a silent bug: the network still runs, but its features are degraded because the first layer sees inputs on an unexpected scale.

For training we use `RandomResizedCrop(224)` (random scale and aspect ratio, a strong and standard augmentation) and horizontal flips. For evaluation: resize the short side to 256, then take the central $224\times224$ crop.

In [ ]:
IMNET_MEAN, IMNET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.5, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMNET_MEAN, IMNET_STD),
])
eval_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(IMNET_MEAN, IMNET_STD),
])

train_set = datasets.Flowers102("./data", split="train", transform=train_tf, download=True)
val_set = datasets.Flowers102("./data", split="val", transform=eval_tf, download=True)
test_set = datasets.Flowers102("./data", split="test", transform=eval_tf, download=True)
NUM_CLASSES = 102
print(len(train_set), "train |", len(val_set), "val |", len(test_set), "test")

kw = dict(num_workers=2, pin_memory=torch.cuda.is_available())
train_loader = DataLoader(train_set, batch_size=32, shuffle=True, **kw)
val_loader = DataLoader(val_set, batch_size=128, **kw)
test_loader = DataLoader(test_set, batch_size=128, **kw)

In [ ]:
def denorm(t):
    return (t * torch.tensor(IMNET_STD).view(3, 1, 1) + torch.tensor(IMNET_MEAN).view(3, 1, 1)).clamp(0, 1).permute(1, 2, 0)

fig, axes = plt.subplots(2, 8, figsize=(15, 4))
for ax, i in zip(axes.flat, np.random.choice(len(val_set), 16, replace=False)):
    x, y = val_set[i]
    ax.imshow(denorm(x)); ax.set_title(f"class {y}", fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

## 4. A shared training loop

Same structure as before, with AMP on GPU. One detail matters for feature extraction: we pass `frozen_bn=True` to keep BatchNorm layers of a frozen backbone in **eval mode** during training.

**Why?** `requires_grad=False` stops BatchNorm's $\gamma, \beta$ from being updated, but in `train()` mode BatchNorm still *updates its running mean and variance* from the current batches and normalises with batch statistics. With small batches from a new dataset, that silently changes the "frozen" features. Keeping those layers in eval mode makes the backbone truly fixed.

In [ ]:
use_amp = device.type == "cuda"

def set_bn_eval(module):
    for m in module.modules():
        if isinstance(m, nn.BatchNorm2d):
            m.eval()

def run_epoch(model, loader, optimizer=None, scheduler=None, scaler=None, frozen_bn=False):
    train = optimizer is not None
    model.train(train)
    if train and frozen_bn:
        set_bn_eval(model)
    total_loss, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                logits = model(x)
                loss = F.cross_entropy(logits, y, label_smoothing=0.1 if train else 0.0)
            if train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer); scaler.update()
                if scheduler is not None: scheduler.step()
            total_loss += loss.item() * x.size(0)
            correct += (logits.argmax(1) == y).sum().item(); n += x.size(0)
    return total_loss / n, correct / n

def fit(model, epochs, optimizer, scheduler=None, frozen_bn=False, label=""):
    scaler = torch.amp.GradScaler(enabled=use_amp)
    hist, best, best_state = {"train_acc": [], "val_acc": []}, 0.0, None
    for ep in range(1, epochs + 1):
        t0 = time.time()
        _, ta = run_epoch(model, train_loader, optimizer, scheduler, scaler, frozen_bn)
        _, va = run_epoch(model, val_loader)
        hist["train_acc"].append(ta); hist["val_acc"].append(va)
        if va > best: best, best_state = va, copy.deepcopy(model.state_dict())
        print(f"[{label}] ep {ep:2d} | train acc {ta:.3f} | val acc {va:.3f} | {time.time() - t0:.0f}s")
    model.load_state_dict(best_state)
    return hist

def count_trainable(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

results = {}

We also use **label smoothing** (0.1) in the training loss: the target distribution becomes $0.9$ on the true class and $0.1/K$ spread over the others ($K$ = number of classes). This discourages over-confident logits and usually helps a little with few examples per class.

## 5. Strategy 0: training from scratch (the baseline)

Same architecture, **random** weights. This tells us how much the pretraining is actually worth.

In [ ]:
EPOCHS_SCRATCH = 15
scratch = models.resnet18(weights=None, num_classes=NUM_CLASSES).to(device)
opt = torch.optim.AdamW(scratch.parameters(), lr=1e-3, weight_decay=0.05)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=1e-3, epochs=EPOCHS_SCRATCH, steps_per_epoch=len(train_loader))
print("trainable params:", f"{count_trainable(scratch):,}")
results["from scratch"] = fit(scratch, EPOCHS_SCRATCH, opt, sched, label="scratch")

Expect poor validation accuracy (often 15 to 30%) despite training accuracy climbing: 10 images per class are not enough to learn visual features from nothing.

## 6. Strategy 1: feature extraction

Steps:

1. Load the pretrained model.
2. **Freeze** every parameter: `p.requires_grad_(False)`. Autograd then does not compute their gradients, which also saves memory and time.
3. **Replace the head**: `model.fc = nn.Linear(512, 102)`. A newly created layer has `requires_grad=True` by default, so it is the only trainable part.
4. Give the optimizer only the trainable parameters.

Because only a linear layer is learned, we can use a fairly large learning rate.

In [ ]:
EPOCHS_FE = 10
fe_model = models.resnet18(weights=weights)
for p in fe_model.parameters():
    p.requires_grad_(False)
fe_model.fc = nn.Linear(fe_model.fc.in_features, NUM_CLASSES)
fe_model = fe_model.to(device)
print("trainable params:", f"{count_trainable(fe_model):,}", "of", f"{sum(p.numel() for p in fe_model.parameters()):,}")

opt = torch.optim.AdamW(fe_model.fc.parameters(), lr=3e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=3e-3, epochs=EPOCHS_FE, steps_per_epoch=len(train_loader))
results["feature extraction"] = fit(fe_model, EPOCHS_FE, opt, sched, frozen_bn=True, label="feat-extract")

A single linear layer on top of frozen ImageNet features typically reaches **80 to 88%** validation accuracy, in a fraction of the time. That gap versus scratch is the value of pretraining.

## 7. Strategy 2: fine-tuning with discriminative learning rates

Now we let the whole network adapt. Two important precautions:

1. **Start from a good head.** If the backbone is updated while the head is still random, the large, noisy gradients from the random head can wreck the pretrained features in the first steps. Starting from the feature-extraction head (Strategy 1) avoids this. (Alternatively: train the head alone for a few epochs, then unfreeze, a two-phase schedule.)
2. **Discriminative learning rates** (Howard and Ruder, 2018, ULMFiT): the general early layers need only small changes, the specific later layers and the head need more. We use parameter groups with learning rates increasing with depth.

$$
\eta_\text{stem} < \eta_\text{layer1} < \dots < \eta_\text{layer4} < \eta_\text{head}.
$$

In [ ]:
EPOCHS_FT = 10
ft_model = copy.deepcopy(fe_model)   # start from the trained feature-extraction head
for p in ft_model.parameters():
    p.requires_grad_(True)

base_lr = 1e-3
groups = [
    {"params": list(ft_model.conv1.parameters()) + list(ft_model.bn1.parameters()), "lr": base_lr / 100},
    {"params": ft_model.layer1.parameters(), "lr": base_lr / 100},
    {"params": ft_model.layer2.parameters(), "lr": base_lr / 30},
    {"params": ft_model.layer3.parameters(), "lr": base_lr / 10},
    {"params": ft_model.layer4.parameters(), "lr": base_lr / 3},
    {"params": ft_model.fc.parameters(), "lr": base_lr},
]
opt = torch.optim.AdamW(groups, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[g["lr"] for g in groups],
                                            epochs=EPOCHS_FT, steps_per_epoch=len(train_loader))
print("trainable params:", f"{count_trainable(ft_model):,}")
for g in opt.param_groups:
    print(f"  group lr (peak): {g['max_lr']:.1e}")
results["fine-tuning"] = fit(ft_model, EPOCHS_FT, opt, sched, frozen_bn=False, label="fine-tune")

Here we let BatchNorm update its statistics (`frozen_bn=False`), since the whole network is adapting to the new data. With very small batches (< 16), it can be better to keep BN frozen even while fine-tuning.

## 8. Comparison on the test set

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for name, h in results.items():
    ax.plot(range(1, len(h["val_acc"]) + 1), h["val_acc"], "o-", label=f"{name} (val)")
ax.set_xlabel("epoch"); ax.set_ylabel("validation accuracy"); ax.legend(); ax.set_ylim(0, 1)
plt.show()

models_by_name = {"from scratch": scratch, "feature extraction": fe_model, "fine-tuning": ft_model}
print(f"{'strategy':<22}{'trainable params':>18}{'test acc':>10}")
test_acc = {}
for name, m in models_by_name.items():
    _, acc = run_epoch(m, test_loader)
    test_acc[name] = acc
    print(f"{name:<22}{count_trainable(m):>18,}{acc:>10.4f}")

Typical ranking: **fine-tuning > feature extraction >> scratch** (roughly 90%, 85%, 25% on test). With 10 examples per class, pretraining is the difference between an unusable model and a good one.

## 9. Inspecting the embedding space with t-SNE

The 512-dimensional output of the global average pool is the image's **embedding**. If transfer works, images of the same class should cluster in this space even though the backbone never saw flowers labelled by species.

**t-SNE** (van der Maaten and Hinton, 2008) maps high-dimensional points to 2D while trying to preserve *neighbourhoods*: points close in 512-D stay close in 2D. Distances between far-apart clusters and cluster sizes in a t-SNE plot are not meaningful, only local groupings are.

We compare embeddings from (a) a randomly initialised ResNet, (b) the pretrained ImageNet ResNet with no training on flowers, and (c) our fine-tuned model, on the validation images of 15 classes.

In [ ]:
from sklearn.manifold import TSNE

@torch.no_grad()
def embed(model, loader):
    trunk = nn.Sequential(*list(model.children())[:-1]).to(device).eval()  # everything except fc
    feats, labels = [], []
    for x, y in loader:
        feats.append(trunk(x.to(device)).flatten(1).cpu()); labels.append(y)
    return torch.cat(feats).numpy(), torch.cat(labels).numpy()

subset_classes = list(range(15))
val_labels_all = np.array([val_set[i][1] for i in range(len(val_set))]) if not hasattr(val_set, "_labels") else np.array(val_set._labels)
idx = [i for i, y in enumerate(val_labels_all) if y in subset_classes]
sub_loader = DataLoader(torch.utils.data.Subset(val_set, idx), batch_size=128)

encoders = {"random init": models.resnet18(weights=None),
            "ImageNet pretrained (no flower training)": models.resnet18(weights=weights),
            "fine-tuned on flowers": ft_model}
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
for ax, (name, m) in zip(axes, encoders.items()):
    f, l = embed(m, sub_loader)
    z = TSNE(n_components=2, perplexity=min(20, len(f) - 1), init="pca", random_state=0).fit_transform(f)
    ax.scatter(z[:, 0], z[:, 1], c=l, cmap="tab20", s=14)
    ax.set_title(name); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

Random features give a shapeless cloud. The **pretrained** ImageNet features, without ever being trained on these labels, already form clear clusters. That is why a linear head is enough. Fine-tuning tightens them further.

## 10. Looking at predictions and errors

In [ ]:
@torch.no_grad()
def predict_all(model, loader):
    model.eval(); P, Y = [], []
    for x, y in loader:
        P.append(F.softmax(model(x.to(device)), 1).cpu()); Y.append(y)
    return torch.cat(P), torch.cat(Y)

probs, labels = predict_all(ft_model, test_loader)
preds = probs.argmax(1)
per_class = torch.stack([(preds[labels == c] == c).float().mean() for c in range(NUM_CLASSES)])
worst = per_class.argsort()[:5]
print("hardest classes:", [(c.item(), round(per_class[c].item(), 2)) for c in worst])

wrong = (preds != labels).nonzero().squeeze(1)
sel = wrong[torch.randperm(len(wrong))[:8]]
fig, axes = plt.subplots(1, 8, figsize=(17, 2.8))
for ax, i in zip(axes, sel):
    x, y = test_set[i]
    ax.imshow(denorm(x)); ax.axis("off")
    ax.set_title(f"true {y}\npred {preds[i].item()} ({probs[i].max():.2f})", fontsize=9)
plt.suptitle("Random test mistakes"); plt.tight_layout(); plt.show()

## 11. Saving and single-image inference

For deployment, save the `state_dict` together with what you need to rebuild the model (architecture name, number of classes, preprocessing). At inference time, apply the **evaluation** transform, add a batch dimension, use `eval()` and `no_grad()`.

In [ ]:
torch.save({"state_dict": ft_model.state_dict(), "num_classes": NUM_CLASSES, "arch": "resnet18"},
           "flowers_resnet18.pt")

def load_model(path):
    ckpt = torch.load(path, map_location=device)
    m = models.resnet18(weights=None)
    m.fc = nn.Linear(m.fc.in_features, ckpt["num_classes"])
    m.load_state_dict(ckpt["state_dict"])
    return m.to(device).eval()

@torch.no_grad()
def predict_image(model, pil_image, k=3):
    x = eval_tf(pil_image).unsqueeze(0).to(device)        # (1, 3, 224, 224)
    p = F.softmax(model(x), 1)[0]
    top = p.topk(k)
    return [(i.item(), round(v.item(), 3)) for v, i in zip(top.values, top.indices)]

deployed = load_model("flowers_resnet18.pt")
raw_test = datasets.Flowers102("./data", split="test", download=False)   # PIL images, no transform
pil, true_label = raw_test[0]
print("true:", true_label, "| top-3 (class, prob):", predict_image(deployed, pil))

# To use your own photo in Colab:
# from google.colab import files; from PIL import Image
# up = files.upload(); pil = Image.open(next(iter(up))).convert("RGB"); print(predict_image(deployed, pil))

## Summary

* Pretrained CNNs learn general-to-specific features; early and middle layers transfer across natural-image tasks.
* **Feature extraction:** freeze the backbone (`requires_grad_(False)`), replace `fc`, train only the head. Keep frozen BatchNorm layers in eval mode.
* **Fine-tuning:** unfreeze, start from a trained head, use small and **discriminative** learning rates.
* Always use the preprocessing the backbone was trained with (ImageNet mean/std, 224 crops).
* With 10 images per class: scratch ≈ useless, feature extraction ≈ good, fine-tuning ≈ best.
* t-SNE of embeddings shows that pretrained features already cluster unseen classes.

## Exercises

1. **Partial fine-tuning.** Unfreeze only `layer4` + `fc`. Where does it land between feature extraction and full fine-tuning, in accuracy and in training time?
2. **Uniform LR.** Fine-tune with the same learning rate `1e-3` for all layers. Then try `1e-2`. What happens to validation accuracy in the first epoch, and why?
3. **Cold head.** Fine-tune starting from a *random* head instead of `fe_model`. Compare the first few epochs.
4. **Other backbones.** Swap in `models.efficientnet_b0` or `models.convnext_tiny` (note: their head is `classifier`, not `fc`). Which gives the best accuracy per second of training?
5. **Linear probe with scikit-learn.** Extract embeddings once with `embed`, then fit `sklearn.linear_model.LogisticRegression` on the train embeddings. How close do you get to Strategy 1, in seconds?
6. **Domain shift.** Repeat feature extraction on a dataset far from ImageNet, such as `datasets.EuroSAT` (satellite) or a medical MedMNIST dataset. Does the gap between feature extraction and fine-tuning grow? Relate this to the general-to-specific picture of Section 1.